In [1]:
import os
import cv2
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models


class DBlock(nn.Module):
    def __init__(self, channels):
        super(DBlock, self).__init__()
        self.dilate1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, dilation=1)
        self.dilate2 = nn.Conv2d(channels, channels, kernel_size=3, padding=2, dilation=2)
        self.dilate3 = nn.Conv2d(channels, channels, kernel_size=3, padding=4, dilation=4)
        self.dilate4 = nn.Conv2d(channels, channels, kernel_size=3, padding=8, dilation=8)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        d1 = self.relu(self.dilate1(x))
        d2 = self.relu(self.dilate2(d1))
        d3 = self.relu(self.dilate3(d2))
        d4 = self.relu(self.dilate4(d3))

        return x + d1 + d2 + d3 + d4
    
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DecoderBlock, self).__init__()
        mid_channels = in_channels // 4

        self.conv1 = nn.Conv2d(in_channels, mid_channels, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(mid_channels)
        self.relu1 = nn.ReLU(inplace=True)

        # Simulate ExpandDims + Conv3DTranspose + Squeeze
        self.deconv3d = nn.ConvTranspose3d(
            mid_channels, mid_channels,
            kernel_size=(1, 3, 3),
            stride=(1, 2, 2),
            padding=(0, 1, 1),
            output_padding=(0, 1, 1)
        )

        self.bn2 = nn.BatchNorm2d(mid_channels)
        self.relu2 = nn.ReLU(inplace=True)

        self.conv3 = nn.Conv2d(mid_channels, out_channels, kernel_size=1)
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.relu3 = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)

        # Expand dims
        x = x.unsqueeze(2)  # (B, C, 1, H, W)
        x = self.deconv3d(x)
        x = x.squeeze(2)    # (B, C, H*2, W*2)

        x = self.bn2(x)
        x = self.relu2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)

        return x

class DLinkNet34(nn.Module):
    def __init__(self, num_classes=1, pretrained=True):
        super(DLinkNet34, self).__init__()
        filters = [64, 128, 256, 512]

        resnet = models.resnet34(pretrained=pretrained)

        # Encoder
        self.firstconv = resnet.conv1
        self.firstbn = resnet.bn1
        self.firstrelu = resnet.relu
        self.firstmaxpool = resnet.maxpool

        self.encoder1 = resnet.layer1
        self.encoder2 = resnet.layer2
        self.encoder3 = resnet.layer3
        self.encoder4 = resnet.layer4

        # Center
        self.dblock = DBlock(512)

        # Decoder
        self.decoder4 = DecoderBlock(filters[3], filters[2])
        self.decoder3 = DecoderBlock(filters[2], filters[1])
        self.decoder2 = DecoderBlock(filters[1], filters[0])
        self.decoder1 = DecoderBlock(filters[0], filters[0])

        # Final conv
        self.finaldeconv1 = nn.ConvTranspose2d(filters[0], 32, kernel_size=4, stride=2, padding=1)
        self.finalrelu1 = nn.ReLU(inplace=True)
        self.finalconv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.finalrelu2 = nn.ReLU(inplace=True)
        self.finalconv3 = nn.Conv2d(32, num_classes, kernel_size=3, padding=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Encoder
        x = self.firstconv(x)
        x = self.firstbn(x)
        x = self.firstrelu(x)
        x = self.firstmaxpool(x)

        e1 = self.encoder1(x)
        e2 = self.encoder2(e1)
        e3 = self.encoder3(e2)
        e4 = self.encoder4(e3)

        # Center
        center = self.dblock(e4)

        # Decoder
        d4 = self.decoder4(center) + e3
        d3 = self.decoder3(d4) + e2
        d2 = self.decoder2(d3) + e1
        d1 = self.decoder1(d2)

        out = self.finaldeconv1(d1)
        out = self.finalrelu1(out)
        out = self.finalconv2(out)
        out = self.finalrelu2(out)
        out = self.finalconv3(out)

        return out

In [3]:
model_path = '/kaggle/input/models/vuongtran11233/dlinknet-test/pytorch/default/1/best_dlinknet.pth'
csv_path = '/kaggle/input/datasets/vuongtran11233/new-division/data_division.csv'
image_root = '/kaggle/input/datasets/vuongtran11233/combined-dataset/combined_dataset/images'
mask_root = '/kaggle/input/datasets/vuongtran11233/combined-dataset/combined_dataset/masks'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
threshold = 0.5

In [4]:
# === Load model ===
model = DLinkNet34(num_classes=1)
# 1. Load the dictionary file
checkpoint = torch.load(model_path, map_location=device)

# Handle cases where you saved the entire checkpoint dict or just the state_dict
state_dict = checkpoint['model_state_dict']

# 2. Create a clean dictionary without the "module." prefix
from collections import OrderedDict
clean_state_dict = OrderedDict()

for key, value in state_dict.items():
    # If it starts with module., strip it out
    if key.startswith('module.'):
        clean_name = key[7:]  # 'module.' is 7 characters long
        clean_state_dict[clean_name] = value
    else:
        clean_state_dict[key] = value

# 3. Load the clean state dict into your model
model.load_state_dict(clean_state_dict)

#model = DLinkNet34(num_classes=1)
#checkpoint = torch.load(model_path, map_location=device)
#model.load_state_dict(checkpoint['model_state_dict'])

model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 195MB/s]


DLinkNet34(
  (firstconv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (firstbn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (firstrelu): ReLU(inplace=True)
  (firstmaxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (encoder1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu

In [5]:
# === Load file ảnh
df = pd.read_csv(csv_path)
test_df = df[df['split'] == 'test']

# Lấy danh sách file ảnh và mask
image_filenames = test_df['filename'].tolist()
mask_filenames = test_df['maskname'].tolist()

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader

# ==========================================
# 1. DEFINE A FAST MULTI-THREADED DATASET
# ==========================================
class RoadEvaluationDataset(Dataset):
    def __init__(self, img_filenames, mask_filenames, img_root, mask_root):
        self.img_filenames = img_filenames
        self.mask_filenames = mask_filenames
        self.img_root = img_root
        self.mask_root = mask_root

    def __len__(self):
        return len(self.img_filenames)

    def __getitem__(self, idx):
        # Generate full paths
        img_path = os.path.join(self.img_root, self.img_filenames[idx])
        mask_path = os.path.join(self.mask_root, self.mask_filenames[idx])

        # Read Image
        image = cv2.imread(img_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Normalize and transform Image (matching training)
        input_tensor = image_rgb.astype(np.float32) / 255.0
        input_tensor = input_tensor * 3.2 - 1.6
        input_tensor = np.transpose(input_tensor, (2, 0, 1)) # (C, H, W)

        # Read and process Mask (matching training normalization perfectly)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask_normalized = mask.astype(np.float32) / 255.0
        mask_binary = (mask_normalized > 0.5).astype(np.float32)

        return torch.tensor(input_tensor), torch.tensor(mask_binary).unsqueeze(0)


# ==========================================
# 2. INITIALIZE DATALOADER & CONTAINERS
# ==========================================
eval_dataset = RoadEvaluationDataset(image_filenames, mask_filenames, image_root, mask_root)

# Using batch_size=16 or 32 utilizes the T4 GPU efficiently.
# num_workers=2 enables asynchronous background image loading on CPU threads.
eval_loader = DataLoader(
    eval_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [13]:
import numpy as np
import torch
import torch.nn.functional as F
from skimage.morphology import skeletonize

def get_batch_iou_dice_counts(pred_mask, gt_mask):
    """Returns the raw intersection and total pixels to compute running IoU/Dice."""
    pred_flat = pred_mask.view(-1).float()
    gt_flat = gt_mask.view(-1).float()
    
    intersection = torch.sum(pred_flat * gt_flat).item()
    total_pixels = (torch.sum(pred_flat) + torch.sum(gt_flat)).item()
    return intersection, total_pixels


def compute_batch_cldice(pred_mask, gt_mask):
    """Computes clDice for a batch by converting tensors to numpy item-by-item."""
    # Move to CPU and convert to numpy immediately to free GPU/RAM references
    P = pred_mask.detach().cpu().numpy().astype(bool)
    G = gt_mask.detach().cpu().numpy().astype(bool)
    
    # If 3D or 4D batch, squeeze out extra dimensions
    P = np.squeeze(P)
    G = np.squeeze(G)
    
    # Handle batch dimension if present after squeeze
    if P.ndim == 2:
        P, G = np.expand_dims(P, 0), np.expand_dims(G, 0)
        
    batch_scores = []
    for i in range(P.shape[0]):
        p_img, g_img = P[i], G[i]
        
        S_P = skeletonize(p_img) if np.sum(p_img) > 0 else np.zeros_like(p_img)
        S_G = skeletonize(g_img) if np.sum(g_img) > 0 else np.zeros_like(g_img)
        
        if np.sum(S_P) == 0 and np.sum(S_G) == 0:
            batch_scores.append(1.0)
            continue
            
        t_prec = np.sum(S_P & g_img) / np.sum(S_P) if np.sum(S_P) > 0 else 0.0
        t_sens = np.sum(S_G & p_img) / np.sum(S_G) if np.sum(S_G) > 0 else 0.0
        
        score = (2.0 * t_prec * t_sens) / (t_prec + t_sens) if (t_prec + t_sens) > 0 else 0.0
        batch_scores.append(score)
        
    return np.mean(batch_scores)

In [14]:
# 1. Initialize running accumulators (Takes virtually 0 RAM)
total_intersection = 0
total_pixels_sum = 0
biou_running_sum = 0
cldice_running_sum = 0
total_batches = 0

model.eval()
with torch.no_grad():
    for batch_imgs, batch_masks in tqdm(eval_loader, total=len(eval_loader)):
        # Move input batch to GPU asynchronously
        batch_imgs = batch_imgs.to(device, non_blocking=True)

        # Forward pass on GPU
        outputs = model(batch_imgs)

        # Threshold directly on the GPU (convert to binary 0 or 1 integer)
        preds = (outputs > threshold).int()
        batch_masks = batch_masks.int()

        # 2. CALCULATE RUNNING METRICS ON THE FLY
        # (This uses the batch-optimized functions from earlier)
        inter, total_px = get_batch_iou_dice_counts(preds, batch_masks.to(device))
        total_intersection += inter
        total_pixels_sum += total_px
        
        cldice_running_sum += compute_batch_cldice(preds, batch_masks) # Moves to CPU internally
        
        total_batches += 1

# ==========================================
# 3. FINAL DATASET-WIDE REDUCTION
# ==========================================
final_iou = total_intersection / (total_pixels_sum - total_intersection + 1e-6)
final_dice = (2.0 * total_intersection) / (total_pixels_sum + 1e-6)
final_cldice = cldice_running_sum / total_batches

print(f"Road IoU: {final_iou:.4f} | Road Dice: {final_dice:.4f} | clDice: {final_cldice:.4f}")

100%|██████████| 70/70 [02:51<00:00,  2.45s/it]

Road IoU: 0.6313 | Road Dice: 0.7739 | clDice: 0.8245
